# Passos Mágicos: limpeza e unificação da base PEDE

Neste notebook tratamos as bases da Pesquisa Extensiva do Desenvolvimento Educacional (PEDE) de 2022, 2023 e 2024 e juntamos tudo em uma única tabela, usada depois na análise exploratória e no modelo.

A entrada é o arquivo `BASE DE DADOS PEDE 2024 - DATATHON.xlsx`, fornecido pela Passos Mágicos, com uma aba por ano. A saída é `dados/tratados/pede_long.csv`, com uma linha por aluno (`ra`) em cada ano.

## 1. Importação das bibliotecas

In [1]:
%pip install "pandas>=2.2" numpy openpyxl -q

In [2]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARQ_BRUTO = RAIZ / "dados" / "BASE DE DADOS PEDE 2024 - DATATHON.xlsx"
ARQ_SAIDA = RAIZ / "dados" / "tratados" / "pede_long.csv"

## 2. Aquisição dos dados

In [3]:
abas = pd.read_excel(ARQ_BRUTO, sheet_name=None)

for nome, d in abas.items():
    print(f"{nome}: {d.shape[0]} linhas x {d.shape[1]} colunas")

PEDE2022: 860 linhas x 42 colunas
PEDE2023: 1014 linhas x 48 colunas
PEDE2024: 1156 linhas x 50 colunas


In [4]:
todas_colunas = sorted(set().union(*[d.columns for d in abas.values()]))
presenca = pd.DataFrame(
    {nome: [c in d.columns for c in todas_colunas] for nome, d in abas.items()},
    index=todas_colunas,
)
presenca

,PEDE2022,PEDE2023,PEDE2024
Ano ingresso,True,True,True
Ano nasc,True,False,False
Atingiu PV,True,True,True
Ativo/ Inativo,False,False,True
Ativo/ Inativo.1,False,False,True
...,...,...,...
Rec Av2,True,True,True
Rec Av3,True,True,False
Rec Av4,True,True,False
Rec Psicologia,True,True,True


As três abas não seguem o mesmo layout. O mesmo conceito aparece com nomes diferentes (`Defas` e `Defasagem`, `Matem` e `Mat`, `INDE 22`, `INDE 2023` e `INDE 2024`), o IPP não existe em 2022 e as abas de 2023 e 2024 repetem colunas históricas (`INDE 22`, `Pedra 20` etc.).

O dicionário de dados fornecido descreve as colunas das bases de 2020 e 2021. Ele foi usado como referência para os conceitos dos indicadores, mas não para os nomes das colunas.

Antes de descartar as colunas históricas, verificamos se elas são de fato cópias da aba de origem:

In [5]:
base_22 = abas["PEDE2022"][["RA", "INDE 22"]]
for aba in ["PEDE2023", "PEDE2024"]:
    m = base_22.merge(abas[aba][["RA", "INDE 22"]], on="RA").dropna()
    dif_max = (m["INDE 22_x"] - m["INDE 22_y"]).abs().max()
    print(f"{aba}: {len(m)} alunos em comum, maior diferença no INDE 22 = {dif_max:.6f}")

PEDE2023: 600 alunos em comum, maior diferença no INDE 22 = 0.000000
PEDE2024: 472 alunos em comum, maior diferença no INDE 22 = 0.000499


A diferença máxima é menor que 0,0005 e vem apenas do arredondamento: a aba 2022 guarda 3 casas decimais. As colunas históricas são redundantes no formato longo e podem ser descartadas.

## 3. Padronização das colunas

Mantemos a chave do aluno, as variáveis de contexto (fase, idade, gênero, ano de ingresso, instituição), os indicadores do PEDE e as notas que compõem o IDA. Descartamos:

- `Cg`, `Cf` e `Ct`, rankings calculados a partir do INDE. Além de redundantes, vazariam informação para o modelo.
- `Rec Psicologia` e `Indicado`, preenchidas só em 2022.
- `Avaliador*` e `Rec Av*`, que identificam avaliadores e trazem recomendações em texto, com formato diferente a cada ano.
- `Destaque IEG/IDA/IPV`, texto livre que ficaria para uma análise qualitativa, fora do escopo desta etapa.
- `Ativo/ Inativo`, porque todos os registros são "Cursando".
- As colunas históricas, `Nome`, `Turma`, `Nº Av` e as datas de nascimento, que são redundantes ou não usamos.

In [6]:
RENOMEAR = {
    2022: {
        "RA": "ra", "Fase": "fase", "Idade 22": "idade", "Gênero": "genero",
        "Ano ingresso": "ano_ingresso", "Instituição de ensino": "instituicao",
        "INDE 22": "inde", "Pedra 22": "pedra",
        "IAA": "iaa", "IEG": "ieg", "IPS": "ips", "IDA": "ida", "IPV": "ipv", "IAN": "ian",
        "Matem": "mat", "Portug": "por", "Inglês": "ing",
        "Fase ideal": "fase_ideal", "Defas": "defasagem", "Atingiu PV": "atingiu_pv",
    },
    2023: {
        "RA": "ra", "Fase": "fase", "Idade": "idade", "Gênero": "genero",
        "Ano ingresso": "ano_ingresso", "Instituição de ensino": "instituicao",
        "INDE 2023": "inde", "Pedra 2023": "pedra",
        "IAA": "iaa", "IEG": "ieg", "IPS": "ips", "IPP": "ipp", "IDA": "ida", "IPV": "ipv", "IAN": "ian",
        "Mat": "mat", "Por": "por", "Ing": "ing",
        "Fase Ideal": "fase_ideal", "Defasagem": "defasagem", "Atingiu PV": "atingiu_pv",
    },
    2024: {
        "RA": "ra", "Fase": "fase", "Idade": "idade", "Gênero": "genero",
        "Ano ingresso": "ano_ingresso", "Instituição de ensino": "instituicao",
        "INDE 2024": "inde", "Pedra 2024": "pedra",
        "IAA": "iaa", "IEG": "ieg", "IPS": "ips", "IPP": "ipp", "IDA": "ida", "IPV": "ipv", "IAN": "ian",
        "Mat": "mat", "Por": "por", "Ing": "ing",
        "Fase Ideal": "fase_ideal", "Defasagem": "defasagem", "Atingiu PV": "atingiu_pv",
    },
}

COLUNAS = [
    "ra", "ano", "fase", "fase_ideal", "defasagem", "idade", "genero", "ano_ingresso", "instituicao",
    "inde", "pedra", "iaa", "ieg", "ips", "ipp", "ida", "ipv", "ian",
    "mat", "por", "ing", "atingiu_pv",
]
ABA_POR_ANO = {2022: "PEDE2022", 2023: "PEDE2023", 2024: "PEDE2024"}

for ano, mapa in RENOMEAR.items():
    faltando = set(mapa) - set(abas[ABA_POR_ANO[ano]].columns)
    assert not faltando, f"{ano}: colunas não encontradas {faltando}"

In [7]:
# reindex mantém apenas COLUNAS; as ausentes no ano (ex.: ipp em 2022) viram NaN
df = pd.concat(
    [abas[aba].rename(columns=RENOMEAR[ano]).assign(ano=ano).reindex(columns=COLUNAS)
     for ano, aba in ABA_POR_ANO.items()],
    ignore_index=True,
)

print(f"Shape: {df.shape}")
print(df.groupby("ano").size())

Shape: (3030, 22)
ano
2022     860
2023    1014
2024    1156
dtype: int64


In [8]:
df.dtypes

ra                  str
ano               int64
fase             object
fase_ideal          str
defasagem         int64
idade            object
genero              str
ano_ingresso      int64
instituicao         str
inde             object
pedra               str
iaa             float64
ieg             float64
ips             float64
ipp             float64
ida             float64
ipv             float64
ian             float64
mat             float64
por             float64
ing             float64
atingiu_pv       object
dtype: object

`fase`, `idade`, `inde` e `atingiu_pv` ainda estão como texto ou tipo misto, o que indica problemas de formato tratados na próxima seção.

## 4. Tratamento das variáveis

### 4.1 Fase e fase ideal

A fase aparece em três formatos: inteiro em 2022 (`0` a `7`), texto em 2023 (`"FASE 3"`, `"ALFA"`) e, em 2024, junto com a turma (`"3B"`, `"7E"`). A fase ideal vem como texto descritivo (`"Fase 3 (7º e 8º ano)"`). Como a fase tem sempre um único dígito, convertemos as duas colunas para inteiro a partir do primeiro dígito, com ALFA = 0.

In [9]:
def fase_para_int(valor):
    texto = str(valor).strip().upper()
    if texto.startswith("ALFA"):
        return 0
    return int(re.search(r"\d", texto).group())

df["fase"] = df["fase"].map(fase_para_int)
df["fase_ideal"] = df["fase_ideal"].map(fase_para_int)

pd.crosstab(df["fase"], df["ano"])

ano,2022,2023,2024
fase,,,
0,190,231,196
1,192,173,185
2,155,200,185
3,148,132,211
4,76,94,115
5,60,65,100
6,18,33,25
7,21,23,37
8,0,63,64


A fase 8 corresponde aos universitários e a fase 9 só aparece em 2024 (38 alunos, todos sem INDE calculado). O número de alunos na fase ALFA também cresce bastante em 2023 e 2024, o que muda a composição da base entre os anos. Isso precisa ser considerado antes de comparar médias gerais.

### 4.2 Idade

Na aba 2023, parte das idades foi convertida em data pelo Excel (o valor 8 aparece como `1900-01-08`). O dia da data corresponde à idade original.

In [10]:
df["idade"] = df["idade"].map(lambda v: v.day if hasattr(v, "day") else int(v))
df.groupby("ano")["idade"].agg(["min", "median", "max"])

,min,median,max
ano,,,
2022,7,12.0,21
2023,7,12.0,26
2024,7,12.0,27


### 4.3 INDE e Pedra

Em 2024, o INDE e a Pedra trazem o marcador `"INCLUIR"` para os 38 alunos da fase 9. Esse marcador vira `NaN`. A grafia `"Agata"` (2023 e 2024) é padronizada para `"Ágata"`, e a Pedra vira uma categoria ordenada.

In [11]:
ORDEM_PEDRA = ["Quartzo", "Ágata", "Ametista", "Topázio"]

df["inde"] = pd.to_numeric(df["inde"], errors="coerce")
df["pedra"] = df["pedra"].replace({"Agata": "Ágata", "INCLUIR": np.nan})
df["pedra"] = pd.Categorical(df["pedra"], categories=ORDEM_PEDRA, ordered=True)

pd.crosstab(df["pedra"], df["ano"], dropna=False)

ano,2022,2023,2024
pedra,,,
NaN,0,83,102
Quartzo,132,72,112
Ágata,250,246,225
Ametista,348,381,391
Topázio,130,232,326


Pelo dicionário de dados, a Pedra é definida por faixas fixas do INDE. Verificamos se a classificação registrada segue essas faixas:

In [12]:
faixas = [-np.inf, 5.506, 6.868, 8.230, np.inf]
pedra_pelas_faixas = pd.cut(df["inde"], bins=faixas, labels=ORDEM_PEDRA)

validos = df["inde"].notna() & df["pedra"].notna()
concorda = pedra_pelas_faixas[validos].astype(str) == df.loc[validos, "pedra"].astype(str)
concorda.groupby(df.loc[validos, "ano"]).mean().round(3)

ano
2022    0.803
2023    0.832
2024    0.821
dtype: float64

A concordância fica entre 80% e 83%, o que indica que os cortes usados na classificação variam de ano para ano e não são os do dicionário. Mantemos a Pedra original, porque é a classificação que a associação de fato utiliza.

### 4.4 Gênero

In [13]:
df["genero"] = df["genero"].replace({"Menina": "Feminino", "Menino": "Masculino"})
df["genero"].value_counts()

genero
Feminino     1626
Masculino    1404
Name: count, dtype: int64

### 4.5 Instituição de ensino

As categorias mudam a cada ano e são muito granulares: há subcategorias de escola privada com 1 a 9 alunos. Agrupamos em quatro níveis. "Rede Decisão" e "Escola JP II", que só aparecem em 2022, foram classificadas como privadas. No Relatório PEDE 2021 elas aparecem como instituições de ensino fundamental e médio parceiras, separadas da rede pública.

In [14]:
def agrupar_instituicao(valor):
    texto = str(valor).lower()
    if "pública" in texto:
        return "Pública"
    if "privada" in texto or "decisão" in texto or "jp ii" in texto:
        return "Privada"
    if "concluiu" in texto or "universitário" in texto or "formado" in texto:
        return "Concluiu EM / Universitário"
    return "Outros"

df["instituicao"] = df["instituicao"].map(agrupar_instituicao)
pd.crosstab(df["instituicao"], df["ano"])

ano,2022,2023,2024
instituicao,,,
Concluiu EM / Universitário,0,7,20
Outros,0,2,1
Privada,108,196,222
Pública,752,809,913


### 4.6 Ano de ingresso

In [15]:
print(df.groupby("ano")["ano_ingresso"].agg(["min", "max"]))
print(f"\nAlunos com mais de um ano de ingresso registrado: {df.groupby('ra')['ano_ingresso'].nunique().gt(1).sum()}")

       min   max
ano             
2022  2016  2022
2023  2016  2023
2024  2021  2024

Alunos com mais de um ano de ingresso registrado: 201


O mesmo aluno tem anos de ingresso diferentes entre as abas, e em 2024 o menor valor é 2021, embora a base de 2022 tenha alunos que ingressaram em 2016. A informação parece ter sido sobrescrita em 2024. Como o ingresso é um evento fixo, usamos o menor ano registrado para cada aluno. Isso é relevante porque o tempo de programa é uma variável candidata tanto na análise de efetividade quanto no modelo.

In [16]:
df["ano_ingresso"] = df.groupby("ra")["ano_ingresso"].transform("min")

### 4.7 Defasagem

Pela definição da associação, defasagem = fase efetiva − fase ideal. Valores negativos indicam atraso. Recalculamos e comparamos com a coluna original:

In [17]:
defasagem_calc = df["fase"] - df["fase_ideal"]
df.loc[defasagem_calc != df["defasagem"], ["ra", "ano", "fase", "fase_ideal", "defasagem", "ian"]]

,ra,ano,fase,fase_ideal,defasagem,ian
2444,RA-1516,2024,3,3,3,10.0
2452,RA-1519,2024,3,3,3,10.0


Apenas duas linhas de 2024 divergem: fase 3, fase ideal 3 e defasagem registrada +3. Como a fórmula confere em todas as outras 3.028 linhas, tratamos esses casos como erro de digitação e usamos o valor recalculado.

In [18]:
df["defasagem"] = defasagem_calc

### 4.8 IAN

In [19]:
pd.crosstab(df["defasagem"], df["ian"])

ian,2.5,5.0,10.0
defasagem,,,
-5,1,0,0
-4,5,0,0
-3,39,0,0
-2,0,383,0
-1,0,1259,0
0,0,0,1154
1,0,0,165
2,0,0,24


O IAN é uma função determinística da defasagem: 10 para alunos em fase ou adiantados, 5 para atraso de uma ou duas fases e 2,5 para atraso de três fases ou mais. Isso tem duas consequências para o projeto:

1. As três faixas respondem diretamente à pergunta sobre alunos moderada e severamente defasados. Criamos a variável `nivel_defasagem` com esses rótulos.
2. Na modelagem, o IAN, o INDE (que incorpora o IAN), a Pedra (derivada do INDE) e a combinação fase + idade entregam a própria resposta quando o alvo é a defasagem do mesmo ano. Por isso o modelo preditivo usa os indicadores de um ano para prever o risco no ano seguinte.

In [20]:
ian_esperado = np.select([df["defasagem"] >= 0, df["defasagem"] >= -2], [10.0, 5.0], default=2.5)
assert (df["ian"] == ian_esperado).all()

df["nivel_defasagem"] = pd.Categorical(
    np.select([df["defasagem"] >= 0, df["defasagem"] >= -2], ["Em fase", "Moderada"], default="Severa"),
    categories=["Em fase", "Moderada", "Severa"], ordered=True,
)
pd.crosstab(df["nivel_defasagem"], df["ano"], normalize="columns").round(3)

ano,2022,2023,2024
nivel_defasagem,,,
Em fase,0.301,0.456,0.538
Moderada,0.666,0.531,0.459
Severa,0.033,0.014,0.003


### 4.9 Ponto de virada

A flag `Atingiu PV` está preenchida apenas em 2022. Convertemos para 0/1, mas as análises de ponto de virada ao longo do tempo usam a nota do IPV, que existe nos três anos.

In [21]:
df["atingiu_pv"] = df["atingiu_pv"].map({"Sim": 1, "Não": 0})
df.groupby("ano")["atingiu_pv"].agg(["count", "mean"])

,count,mean
ano,,
2022,860,0.131395
2023,0,NaN
2024,0,NaN


### 4.10 Faixa dos indicadores

In [22]:
INDICADORES = ["inde", "iaa", "ieg", "ips", "ipp", "ida", "ipv", "ian"]
df.groupby("ano")[INDICADORES].agg(["min", "max"]).T.unstack()

ano    2022              2023                 2024           
        min     max       min        max       min        max
inde  3.032   9.442  3.745542   9.371200  3.789478   9.531325
iaa   0.000  10.000  0.000000  10.000000  0.000000  10.002000
ieg   0.000  10.000  3.700000  10.000000  0.000000  10.000000
ips   2.500  10.000  2.520000  10.000000  2.510000  10.000000
ipp     NaN     NaN  3.750000   9.791667  2.500000  10.000000
ida   0.000   9.900  0.000000  10.000000  0.000000  10.000000
ipv   2.500  10.000  3.320000  10.010000  2.943333   9.760000
ian   2.500  10.000  2.500000  10.000000  2.500000  10.000000

O IAA de 2024 chega a 10,002 (quase todos os valores desse ano terminam em ,002) e o IPV de 2023 chega a 10,01. São resíduos de arredondamento na origem, e limitamos todos os indicadores ao intervalo de 0 a 10.

In [23]:
df[INDICADORES] = df[INDICADORES].clip(lower=0, upper=10)

## 5. Qualidade dos dados

Os casos abaixo não têm correção segura. Deixamos os valores como estão, marcamos os registros com colunas `flag_*` e tratamos cada caso na análise correspondente.

### 5.1 Universitários

In [24]:
df["flag_universitario"] = df["fase"] >= 8

(df.groupby(["ano", "flag_universitario"])
   .agg(alunos=("ra", "size"),
        inde_nulo=("inde", lambda s: s.isna().mean()),
        ida_nulo=("ida", lambda s: s.isna().mean()),
        ieg_zero=("ieg", lambda s: (s == 0).mean()))
   .round(2))

alunos  inde_nulo  ida_nulo  ieg_zero
ano  flag_universitario                                       
2022 False                  860       0.00      0.00      0.00
2023 False                  951       0.02      0.01      0.00
     True                    63       1.00      1.00      0.00
2024 False                 1054       0.00      0.00      0.01
     True                   102       1.00      0.99      1.00

Em 2023 e 2024 os universitários não têm INDE nem IDA, e em 2024 todos têm IEG = 0. Eles não passam pelo mesmo processo de avaliação das fases 0 a 7. Mantê-los derrubaria as médias de IEG e INDE e criaria uma queda que não existe. As análises de indicadores e o modelo consideram apenas `flag_universitario == False`.

### 5.2 Zeros suspeitos

In [25]:
alunos = ~df["flag_universitario"]

for col in ["iaa", "ieg", "ida"]:
    df[f"flag_{col}_zero"] = df[col].eq(0)

df[alunos].groupby("ano")[["flag_iaa_zero", "flag_ieg_zero", "flag_ida_zero"]].sum()

,flag_iaa_zero,flag_ieg_zero,flag_ida_zero
ano,,,
2022,39,1,6
2023,190,0,1
2024,20,7,16


Há 190 autoavaliações zeradas em 2023 (cerca de 20% dos alunos avaliados), distribuídas por todas as fases. É um volume incompatível com notas reais de autoavaliação, e a hipótese mais provável é questionário não respondido registrado como zero. Mantemos o valor original com a flag e avaliamos a sensibilidade das conclusões sobre o IAA com e sem esses registros.

### 5.3 Dados faltantes

In [26]:
(df[alunos].groupby("ano")[INDICADORES + ["mat", "por", "ing"]]
   .agg(lambda s: s.isna().mean())
   .round(2)
   .T)

ano,2022,2023,2024
inde,0.00,0.02,0.00
iaa,0.00,0.00,0.00
ieg,0.00,0.01,0.00
ips,0.00,0.01,0.00
ipp,1.00,0.01,0.00
ida,0.00,0.01,0.00
ipv,0.00,0.01,0.00
ian,0.00,0.00,0.00
mat,0.00,0.01,0.00
por,0.00,0.01,0.00


Desconsiderando os universitários, os indicadores têm no máximo 2% de faltantes por ano. As exceções são estruturais: o IPP não existe em 2022, e o Inglês falta em 55% a 67% dos registros porque não é avaliado em todas as fases. Por isso o IDA de parte dos alunos é a média de duas notas e o de outros é a média de três, o que exige cuidado ao comparar o IDA entre fases.

## 6. Cobertura longitudinal

A base é um retrato anual. Para acompanhar a evolução do mesmo aluno, é preciso cruzar os anos pelo `ra`.

In [27]:
presente = pd.crosstab(df["ra"], df["ano"]).astype(bool)

print(f"Presentes em 2022 e 2023: {(presente[2022] & presente[2023]).sum()}")
print(f"Presentes em 2023 e 2024: {(presente[2023] & presente[2024]).sum()}")
print(f"Presentes nos 3 anos:     {presente.all(axis=1).sum()}")

for t in [2022, 2023]:
    base_t = df.loc[(df["ano"] == t) & alunos, "ra"]
    permanencia = base_t.isin(df.loc[df["ano"] == t + 1, "ra"]).mean()
    print(f"Permanência {t} -> {t + 1} (fases 0 a 7): {permanencia:.0%}")

Presentes em 2022 e 2023: 600
Presentes em 2023 e 2024: 765
Presentes nos 3 anos:     468
Permanência 2022 -> 2023 (fases 0 a 7): 70%
Permanência 2023 -> 2024 (fases 0 a 7): 74%


Há 1.365 pares de anos consecutivos (600 de 2022 para 2023 e 765 de 2023 para 2024), que formam a base disponível para o modelo preditivo.

A permanência de 70% a 74% também introduz um viés de sobrevivência na avaliação de efetividade. Se os alunos que saem do programa tinham indicadores piores, a média dos que ficam sobe sem que haja melhora individual. Por isso, para medir evolução, comparamos os mesmos alunos de um ano para o outro em vez das médias gerais.

## 7. Validação e exportação

In [28]:
assert df.groupby("ano").size().to_dict() == {2022: 860, 2023: 1014, 2024: 1156}
assert not df.duplicated(["ra", "ano"]).any()
assert df["fase"].between(0, 9).all() and df["fase_ideal"].between(0, 8).all()
assert (df["defasagem"] == df["fase"] - df["fase_ideal"]).all()
assert df["idade"].between(5, 30).all()
assert df["genero"].isin(["Feminino", "Masculino"]).all()
assert df[INDICADORES].min().min() >= 0 and df[INDICADORES].max().max() <= 10

ARQ_SAIDA.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(ARQ_SAIDA, index=False)
print(f"Arquivo salvo: {ARQ_SAIDA.name} ({df.shape[0]} linhas x {df.shape[1]} colunas)")
df.head()

Arquivo salvo: pede_long.csv (3030 linhas x 27 colunas)


,ra,ano,fase,fase_ideal,defasagem,idade,genero,ano_ingresso,instituicao,inde,pedra,iaa,ieg,ips,ipp,ida,ipv,ian,mat,por,ing,atingiu_pv,nivel_defasagem,flag_universitario,flag_iaa_zero,flag_ieg_zero,flag_ida_zero
0,RA-1,2022,7,8,-1,19,Feminino,2016,Pública,5.783,Quartzo,8.3,4.1,5.6,NaN,4.0,7.278,5.0,2.7,3.5,6.0,0.0,Moderada,False,False,False,False
1,RA-2,2022,7,7,0,17,Feminino,2017,Privada,7.055,Ametista,8.8,5.2,6.3,NaN,6.8,6.778,10.0,6.3,4.5,9.7,0.0,Em fase,False,False,False,False
2,RA-3,2022,7,7,0,17,Feminino,2016,Privada,6.591,Ágata,0.0,7.9,5.6,NaN,5.6,7.556,10.0,5.8,4.0,6.9,0.0,Em fase,False,True,False,False
3,RA-4,2022,7,7,0,17,Masculino,2017,Privada,5.951,Quartzo,8.8,4.5,5.6,NaN,5.0,5.278,10.0,2.8,3.5,8.7,0.0,Em fase,False,False,False,False
4,RA-5,2022,7,7,0,17,Feminino,2016,Privada,7.427,Ametista,7.9,8.6,5.6,NaN,5.2,7.389,10.0,7.0,2.9,5.7,0.0,Em fase,False,False,False,False


## Conclusão da etapa de limpeza

Unificamos os nomes das colunas dos três anos e montamos a base em formato longo (aluno × ano). No caminho, convertemos a fase e a fase ideal para inteiro a partir de três formatos diferentes, desfizemos a conversão das idades de 2023 em data, tratamos o marcador `"INCLUIR"` do INDE e da Pedra como ausente e padronizamos grafias. Também usamos o menor ano de ingresso de cada aluno (o de 2024 estava sobrescrito), recalculamos 2 defasagens pela fórmula oficial e limitamos os indicadores ao intervalo de 0 a 10.

Algumas coisas que apareceram na limpeza orientam as próximas etapas:

- O IAN é a defasagem recodificada. Nada derivado dele pode explicar a defasagem do mesmo ano, e por isso o modelo prevê o ano seguinte.
- Os universitários passam por outro processo de avaliação e ficam fora das análises de indicadores.
- Como a base tem um retrato por ano, a evolução "ao longo do ano" só pode ser vista de um ano para o outro.
- Muitos alunos entram na ALFA e de 26% a 30% saem a cada ano. A efetividade precisa ser medida acompanhando os mesmos alunos, porque as médias gerais misturam quem entra e quem sai.
- Há lacunas conhecidas: IPP ausente em 2022, flag de ponto de virada só em 2022 e autoavaliações zeradas em 2023.